# Grounded-R1: first GRPO run on a free GPU (Kaggle or Colab, 1x T4)

Runs the main 0.5B configuration of the repository and brings the result files back.
The notebook only calls the scripts of the repository; see `RUNBOOK.md` for the full plan.

**Not yet executed on a T4.** The mixed-precision overrides below (`float32` weights with
`fp16` autocast) are the standard choice for this card but have not been exercised with this
code. Run the pilot cell first and read its output before starting the long run.

Enable a GPU first: Kaggle *Settings → Accelerator → GPU T4 x1* (and *Internet → On*), or
Colab *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/ALI-AL-MARJANI/open-r1.git
%cd open-r1
!git log --oneline -1

In [ ]:
# The pinned training stack. The preinstalled torch build is kept: it matches the GPU driver.
# Colab ships an old torchao that peft 0.21 refuses to load next to; this project does not use it.
!pip uninstall -y -q torchao
!pip install -q trl==0.29.1 transformers==5.18.0 peft==0.21.2 datasets==5.0.1 accelerate==1.15.0 pyyaml matplotlib
import torch, transformers, trl
print(torch.__version__, transformers.__version__, trl.__version__, torch.cuda.get_device_name(0))

## 1. Pilot: 5 steps

Checks that the run fits in memory and measures the time per step. With `step_time` seconds
per step, the 500-step run takes about `500 * step_time / 3600` hours. Kaggle sessions stop
after 12 hours and Colab sessions earlier, so if the estimate is above ~8 hours use the
low-memory or shorter variant suggested below, and rely on `--resume`.

In [ ]:
T4 = "model.dtype=float32 grpo.bf16=false grpo.fp16=true"
!python scripts/grounded/train_grpo.py --config configs/grounded/grpo_qwen0.5b.yaml --seed 0 \
    --checkpoint_root data/pilot --results_root results/pilot \
    --set {T4} grpo.max_steps=5 grpo.logging_steps=1 grpo.save_steps=1000 grpo.warmup_steps=0

import json
for line in open("results/pilot/grpo_qwen0.5b/seed0/train_log.jsonl"):
    entry = json.loads(line)
    if "step_time" in entry:
        print(entry["step"], f"step_time={entry['step_time']:.1f}s", f"format={entry['rewards/format/mean']:.2f}",
              f"abstention={entry['rewards/abstention_rate/mean']:.2f}", f"loss={entry['loss']:.4f}")
!rm -r data/pilot results/pilot
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

If the pilot runs out of memory, add
`grpo.per_device_train_batch_size=4 grpo.gradient_accumulation_steps=8 grpo.gradient_checkpointing=true`
to the `--set` list (same 32 completions per step, half the sequences per backward pass).
If the loss is `nan`, replace the T4 overrides by `model.dtype=float32 grpo.bf16=false` (no mixed precision).
Record whichever overrides you use: they are saved in `run.json` with the rest of the configuration.

## 2. Full run (seed 0)

Checkpoints are written every 100 steps. If the session stops, rerun the first three cells and
this one with `--resume` added (on Kaggle, save `data/` as a dataset or keep the session's
persistent output; on Colab, mount Drive and pass `--checkpoint_root` to a Drive folder).

In [ ]:
!python scripts/grounded/train_grpo.py --config configs/grounded/grpo_qwen0.5b.yaml --seed 0 --set {T4}

## 3. Curves and evaluation

Evaluates the adapter on the four evaluation sets with the same protocol as the committed
zero-shot baseline, then builds the tables.

In [ ]:
!python scripts/grounded/plot_training.py --run grpo_qwen0.5b
!python scripts/grounded/evaluate.py --run_name grpo_qwen0.5b_seed0 \
    --model Qwen/Qwen2.5-0.5B-Instruct --adapter data/grpo_qwen0.5b/seed0
!python scripts/grounded/make_tables.py
!python scripts/grounded/compare.py --a zero_shot_qwen0.5b --b grpo_qwen0.5b_seed0

In [ ]:
from IPython.display import Image, Markdown, display
display(Image("results/grpo/grpo_qwen0.5b/training_curves.png"))
display(Markdown(open("results/tables/squad_v2_distractors.md").read()))

## 4. Bring the results back

Download `results.zip` (and `adapter.zip` if you want to publish the LoRA adapter), unzip
`results.zip` at the root of your local clone, and commit it with an `exp:` message. Then
replace the corresponding `TBD (run pending)` rows of the README with the generated tables.
Report the numbers whatever they are: a run that does not beat the baseline is a result.

In [ ]:
!zip -q -r results.zip results
!zip -q -r adapter.zip data/grpo_qwen0.5b/seed0 -x "*/checkpoint-*"
!ls -lh results.zip adapter.zip